In [ ]:
#Download the needed packages.
try:
    import wrds
except ImportError:
    !pip install wrds
    import wrds

import numpy as np
import os
import pandas as pd
from pathlib import Path

In [ ]:
def find_project_root(start_path, marker="data"):
    path = Path(start_path).resolve()
    for parent in [path] + list(path.parents):
        if (parent / marker).exists():
            return parent
    raise RuntimeError("Project root not found")

PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)

In [ ]:
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

In [ ]:
sp500 = pd.read_csv(
    DATA_RAW/'sp500_daily_membership.csv',
    index_col=0,
    low_memory=False,   # ← THIS is the key fix
    dtype={
        'permno': str,
        'permco': str,
        'gvkey': str,
        'companyid': str
    },
    parse_dates=['date', 'start', 'ending']
)

sp500['companyid'] = (
    pd.to_numeric(sp500['companyid'], errors='coerce')
      .astype('Int64')      # nullable integer (clean!)
      .astype(str)
)

In [ ]:
sp500.dtypes

In [ ]:
sp500

In [ ]:
sp500_ids = sp500[['permno', 'permco','gvkey', 'iid', 'gvkey_iid']]

In [ ]:
crsp_ids = sp500_ids['permno'].unique()

len(crsp_ids)
crsp_ids

In [ ]:
db = wrds.Connection(wrds_username='joostg2000') #achieve connection
print("Connected")

In [ ]:
db.describe_table('crsp_a_stock','wrds_dsfv2_query')

In [ ]:
import os

# -------- SETTINGS --------
chunk_size = 50  # ~50 firms per query (safe & fast)
output_file = DATA_RAW / "crsp_panel.csv"

if os.path.exists(output_file):
    prev_out = pd.read_csv(output_file)
    os.remove(output_file)

# Ensure permnos are clean
crsp_ids = list(pd.Series(crsp_ids).dropna().unique())

# Split into chunks
permno_chunks = np.array_split(crsp_ids, len(crsp_ids) // chunk_size + 1)

# -------- QUERY LOOP --------
cols = [
    "permno",
    "date",
    'price',
    "ret",
    "retx",
    "vol",
    "shrout",
    "sprtrn",
    "dlycap",
    'dlyprevcap'
]

for i, chunk in enumerate(permno_chunks):

    permno_tuple = tuple(int(x) for x in chunk)

    query = f"""
    SELECT  
        permno,
        dlycaldt AS date,
        dlyprc AS price,
        dlyret AS ret,
        dlyretx AS retx,
        dlyvol AS vol,
        shrout,
        sprtrn,
        dlycap,
        dlyprevcap
    FROM crsp_a_stock.wrds_dsfv2_query
    WHERE dlycaldt BETWEEN '2019-01-01' AND '2025-12-31'
      AND permno IN {permno_tuple}
    ORDER BY permno, dlycaldt
    """

    batch = db.raw_sql(query)

    if batch.empty:
        print(f"Chunk {i+1} empty, skipping...")
        continue

    batch = batch[cols]  

    batch.to_csv(
        output_file,
        mode='a',
        header=(i == 0),
        index=False
    )

    print(f"Chunk {i+1}/{len(permno_chunks)} saved ({len(batch)} rows)")

# -------- LOAD FINAL DATA --------
calls = pd.read_csv(output_file, parse_dates=["date"])
calls["date"] = pd.to_datetime(calls["date"], errors="coerce")

# -------- WHAT YOU WANTED --------
n_days = calls["date"].nunique()

print("\n--- FINAL SUMMARY ---")
print(f"Total rows: {len(calls)}")
print(f"Unique permnos: {calls['permno'].nunique()}")
print(f"Total trading days: {n_days}")
print(f"Date range: {calls['date'].min()} → {calls['date'].max()}")

In [ ]:
print(len(crsp_ids))
print(len(set(crsp_ids)))
print(type(crsp_ids[0]))
print(crsp_ids[:10])

In [ ]:
input_set = set(int(x) for x in crsp_ids)
output_set = set(calls["permno"].astype(int).unique())

print("Input:", len(input_set))
print("Output:", len(output_set))

extra = output_set - input_set
print("Extra permnos:", len(extra))
print(list(extra)[:20])

In [ ]:
print("Closing WRDS connection...")

if "db" in globals():
    try:
        db.close()
        print("SUCCESS: Connection closed")
    except Exception:
        print("FAIL: Connection could not be closed (may already be closed)")

    try:
        del db
        print("SUCCESS: Variable deleted")
    except Exception:
        print("FAIL: Could not delete variable")
else:
    print("FAIL: No db variable found.")